# Regex Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Raw dogma.** Raw strings stop Python from interpreting backslashes, so what you see is what the regex engine gets.

In [ ]:
import re

plain = "\\d+"           # doubled backslashes - hard to read
raw = r"\d+"             # identical value, easy to read
print(plain == raw)      # True

note = "Table 4 needs 3 chairs, room 42 needs 21."
print(re.findall(raw, note))    # ['4', '3', '42', '21']

**2. Words and digits.** `\d` matches one digit, `+` repeats it, `{n}` demands exactly n of them.

In [ ]:
import re

log = "ERROR 404 at 12:01 | ok | ERROR 500 at 13:15"

print(re.findall(r"ERROR \d+", log))   # ['ERROR 404', 'ERROR 500']
print(re.findall(r"\d{2}:\d{2}", log))  # ['12:01', '13:15']

**3. search vs match vs fullmatch.** `search` scans anywhere, `match` only tries position 0, and `fullmatch` demands the ENTIRE string fit.

In [ ]:
import re

log = "ERROR 404 at 12:01 | ok | ERROR 500 at 13:15"

m = re.search(r"\d{2}:\d{2}", log)
print(m.group(), "at index", m.start())     # 12:01 at index 13

print("match saw it?", re.match(r"\d{2}:\d{2}", log) is not None)   # False
# match only tries position 0 - the log starts with "ERROR", not a time.

print(re.fullmatch(r"\d{2}", "40") is not None)    # True
print(re.fullmatch(r"\d{2}", "404") is not None)   # False

## Part 2 — Practice

**4. Character menu.** Brackets offer a menu of allowed characters; a leading `^` flips it into "anything except".

In [ ]:
import re

word = "beautiful day"
vowels = re.findall(r"[aeiou]", word)
print(len(vowels), "vowels:", vowels)    # 6 vowels: ['e', 'a', 'u', 'i', 'u', 'a']

token = "fa3 G7 #zz"
print(re.findall(r"[0-9a-f]", token))    # ['f', 'a', '3', '7']

messy = "Room 42, floor #7!"
print(re.findall(r"[^a-zA-Z ]", messy))  # ['4', '2', ',', '#', '7', '!']

**5. Fixed shapes.** `{n,m}` counts repetitions — shapes turn into readable formulas.

In [ ]:
import re

code = "ID-AB123 2026-08-26 x9 zz"

print(re.findall(r"[A-Z]{2}\d{3}", code))    # ['AB123']
print(re.findall(r"\d{4}-\d{2}-\d{2}", code))  # ['2026-08-26']

**6. Flags change the rules.** Flags adjust matching globally and combine with `|`; `MULTILINE` makes `^` apply at every line break.

In [ ]:
import re

reply = "YES please.\nno thanks\nMAYBE later"
print(re.findall(r"\byes\b|\bno\b|\bmaybe\b", reply, re.IGNORECASE))
# ['YES', 'no', 'MAYBE']

build_log = "INFO boot\nERROR disk full\nINFO login"
print("default ^   :", re.findall(r"^ERROR.*", build_log))
print("MULTILINE ^ :", re.findall(r"^ERROR.*", build_log, re.MULTILINE))
# default ^    : []
# MULTILINE ^  : ['ERROR disk full']

**7. Greedy vs lazy.** Greedy grabs up to the LAST closer; adding `?` stops at the FIRST opportunity.

In [ ]:
import re

row = "<td>Sarah</td><td>Dhaka</td>"

print(re.findall(r"<td>(.*)</td>", row))
# ['Sarah</td><td>Dhaka'] - greedy ran to the LAST closing tag

print(re.findall(r"<td>(.*?)</td>", row))
# ['Sarah', 'Dhaka'] - lazy stopped at the FIRST closing tag

## Part 3 — Challenge

**8. Groups extract fields.** Parentheses capture slices of the match — matching becomes extraction, and `\1 \2 \3` shuffle pieces in replacements.

In [ ]:
import re

entry = "Parcel delivered on 2026-08-26 to Dhaka."

m = re.search(r"(\d{4})-(\d{2})-(\d{2})", entry)
print(m.group(0))     # 2026-08-26
print(m.groups())     # ('2026', '08', '26')

named = re.search(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", entry)
print("month =", named.group("month"))    # month = 08

print(re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3/\2/\1", entry))
# Parcel delivered on 26/08/2026 to Dhaka.

**9. Compile once, validate everywhere.** `re.compile` gives the pattern a NAME and pays off in hot loops; `fullmatch` is the validator's tool.

In [ ]:
import re

USERNAME = re.compile(r"[A-Za-z0-9_]{3,16}")
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[A-Za-z]{2,}")

candidates = ["sarah_k", "ab", "has space", "way_too_long_username_x", "Rafi2026"]
for name in candidates:
    ok = USERNAME.fullmatch(name) is not None
    print(f"{name:<26} {'accepted' if ok else 'REJECTED'}")
# sarah_k accepted | ab REJECTED | 'has space' REJECTED
# way_too_long_username_x REJECTED (23 chars > 16) | Rafi2026 accepted

message = ("Write to sarah.rahman@gmail.com or arif+bhuyian@yahoo.co.uk today. "
           "Fragments like @nope.com or name@ are ignored.")
print(EMAIL.findall(message))
# ['sarah.rahman@gmail.com', 'arif+bhuyian@yahoo.co.uk']